# Storage-Agnostic KG Building

**The idea:** the *same* `Baseline` class — one implementation, no
subclassing or per-backend variants — can store the graph wherever you want:
in memory, in a file, or in a Neo4j database. Only **one thing changes**:
`graph_store_backend`. The pipeline code stays identical.

In this notebook you run the **same pipeline twice**, using **only default
arguments**, and query both results with **identical code**:

1. **Locally** → graph kept in RAM (`networkx`)
2. **In Neo4j** → graph streamed into the database (nothing in Python memory)

**Prerequisites:** the Neo4j cells need a running instance and
`NEO4J_URI` / `NEO4J_USER` / `NEO4J_PASSWORD` in the environment (or `.env`).
Everything uses `Baseline`'s defaults — the semantic chunking/dedup steps
download a small embedding model on first run.


In [4]:
# 0) Fetch the data first — a small *connected* set of Wikipedia articles
#    (enriched with hyperlinks, ready for the pipeline).
from kglab.data import Data, DegreeSampler

Data.download(
    "wikipedia",
    path="data/wikipedia/connected.jsonl",
    sampler=DegreeSampler(count=5, target_degree=3.0),
)

{'dataset': 'wikipedia',
 'path': 'data/wikipedia/connected.jsonl',
 'cached': True,
 'downloaded': 0}

In [5]:
# 1) Build LOCALLY — the graph lives in memory as a networkx graph.
from kglab.pipelines import Baseline

local_pipe = Baseline(graph_store_backend="networkx")  # all defaults
kg_local = local_pipe.execute(
    input_paths=["data/wikipedia/connected.jsonl"],
    output_dir="output/neo4j_upload_tutorial/local",
)

store_local = kg_local["graph_store"]  # NetworkXGraphStore
print(f"local   → {store_local.number_of_nodes()} nodes, {store_local.number_of_edges()} edges")

=== Baseline ===
Input:  [PosixPath('data/wikipedia/connected.jsonl')]
Output: output/neo4j_upload_tutorial/local



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12451.39it/s]


[preprocess] 12 documents → 29 chunks
[document_relation] 12 doc nodes, 46 doc-doc edges (hyperlink)
[chunks] 29 chunk nodes, 29 chunk→document edges, 17 chunk→next_chunk edges


Ontology validation: 11 node labels not in schema
Ontology validation: 46 relation types not in schema


[entities] 1407 entity→chunk edges
[build_kg] 1448 entities → 1036 resolved, 1047 nodes, 2244 edges
[export] → output/neo4j_upload_tutorial/local/
[manifest] → output/neo4j_upload_tutorial/local/run_manifest.yaml

Done — results in output/neo4j_upload_tutorial/local/
local   → 1047 nodes, 2244 edges


In [10]:
# 2) The SAME pipeline — only the backend changes: the graph is streamed
#    straight into Neo4j (no in-memory graph, no file export).
#    Credentials are loaded from .env (NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD)
#    and passed directly to the pipeline — nothing hardcoded here.
import os

from dotenv import load_dotenv

load_dotenv()  # loads NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD from .env

neo4j_pipe = Baseline(
    graph_store_backend="neo4j",
    graph_store_options={
        "uri": os.environ["NEO4J_URI"],
        "user": os.environ["NEO4J_USER"],
        "password": os.environ["NEO4J_PASSWORD"],
        "clear": True,  # True = wipe first, False = append
    },
)
kg_neo4j = neo4j_pipe.execute(
    input_paths=["data/wikipedia/connected.jsonl"],
    output_dir="output/neo4j_upload_tutorial/neo4j",
)

store_neo4j = kg_neo4j["graph_store"]  # live Neo4jGraphStore
print("streamed to Neo4j:", kg_neo4j.get("neo4j_stats"))
print(f"neo4j    → {store_neo4j.number_of_nodes()} nodes, {store_neo4j.number_of_edges()} edges")

=== Baseline ===
Input:  [PosixPath('data/wikipedia/connected.jsonl')]
Output: output/neo4j_upload_tutorial/neo4j



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7473.26it/s]


[preprocess] 12 documents → 29 chunks
[document_relation] 12 doc nodes, 46 doc-doc edges (hyperlink)
[chunks] 29 chunk nodes, 29 chunk→document edges, 17 chunk→next_chunk edges
[entities] 1407 entity→chunk edges
[build_kg] streamed 1036 entities / 2424 triples to Neo4j (1036 nodes, 2424 edges)
[build_kg] 1448 entities → 1036 resolved, 0 nodes, 0 edges
[export] no in-memory graph (stored in the backend) — skipping file export
[manifest] → output/neo4j_upload_tutorial/neo4j/run_manifest.yaml

Done — results in output/neo4j_upload_tutorial/neo4j/
streamed to Neo4j: {'nodes_written': 1036, 'edges_written': 2424}
neo4j    → 1047 nodes, 2244 edges


In [11]:
# 3) The read side is storage-agnostic too: both stores answer the same
#    GraphStore API, so downstream code never cares where the graph lives.
for label, store in [("local", store_local), ("neo4j", store_neo4j)]:
    print(
        f"{label:6} {type(store).__name__:20} "
        f"{store.number_of_nodes():>4} nodes, {store.number_of_edges():>4} edges"
    )

first_node = next(iter(store_local.nodes()))
print(
    "\nsame API — has_node  :",
    store_local.has_node(first_node),
    "|",
    store_neo4j.has_node(first_node),
)
print(
    "same API — out_degree:",
    store_local.out_degree(first_node),
    "|",
    store_neo4j.out_degree(first_node),
)

local  NetworkXGraphStore   1047 nodes, 2244 edges
neo4j  Neo4jGraphStore      1047 nodes, 2244 edges

same API — has_node  : True | True
same API — out_degree: 0 | 0


## How the abstraction works

- **Write side** — one routine (`build_kg_into`) builds the graph through any
  `GraphWriter`. Adding a new backend = implementing 5 methods
  (`merge_document`, `merge_chunk`, `merge_entity`, `merge_edge`,
  `merge_structural_edge`).
- **Read side** — every backend exposes the same `GraphStore` API
  (`nodes`, `edges`, `get_node`, `successors`, `search`, `to_networkx`), so
  downstream code never changes. Get one from `kg["graph_store"]` or
  `create_graph_store(backend)`.

## Choosing a backend

| `graph_store_backend` | Graph lives in | Python memory |
|---|---|---|
| `"networkx"` (default) | RAM | whole graph |
| `"sqlite"` | `.db` file | queried data only |
| `"neo4j"` | Neo4j database | none |
| `"json"` / `"graphml"` | exported file | lazy via file |

- With `"neo4j"`, `kg["graph"]` is `None` and no file is exported. Use
  `graph_store_options={"clear": True}` to wipe first, `False` to append.
- For corpora too large to preprocess in RAM, use
  `StreamingPipeline(...).execute_batched(batch_size=N)` (at the cost of
  cross-batch dedup and entity resolution).
- Inspect the local graph as a plain object: `store_local.to_networkx()`.
